In [ ]:

import os
import glob
import subprocess
import shutil

# Path to your main folder and parameter file
main_folder = '/Volumes/T9/DNA-PAINT'
params_file = '/Volumes/T9/DNA-PAINT/raw/localizations_manual/RawFramesforTraining_P5-IS_20pM_locs.yaml'

for tif_path in glob.glob(os.path.join(main_folder, '**', '*.tif'), recursive=True):
    tif_dir = os.path.dirname(tif_path)
    localizations_folder = os.path.join(tif_dir, 'localizations')
    os.makedirs(localizations_folder, exist_ok=True)
    
    # Prepare Localize.yaml in tif folder
    yml_target = os.path.join(tif_dir, 'Localize.yaml')
    if not os.path.exists(yml_target):
        shutil.copy(params_file, yml_target)
        print(f"Copied {params_file} to {yml_target}")
    else:
        print(f"Using existing {yml_target}")

    # Run picasso localize (output will appear in tif_dir)
    print('Processing:', tif_path)
    subprocess.run(['picasso', 'localize', tif_path], check=True)

    # Correct output filename with _locs.hdf5
    basename = os.path.splitext(os.path.basename(tif_path))[0]
    hdf5_name = f"{basename}_locs.hdf5"
    hdf5_source = os.path.join(tif_dir, hdf5_name)
    hdf5_dest = os.path.join(localizations_folder, hdf5_name)

  
    # Correct output filename with _locs.hdf5 and _locs.yaml
    basename = os.path.splitext(os.path.basename(tif_path))[0]
    hdf5_name = f"{basename}_locs.hdf5"
    yaml_name = f"{basename}_locs.yaml"

    hdf5_source = os.path.join(tif_dir, hdf5_name)
    yaml_source = os.path.join(tif_dir, yaml_name)

    hdf5_dest = os.path.join(localizations_folder, hdf5_name)
    yaml_dest = os.path.join(localizations_folder, yaml_name)

    # Move the HDF5 file
    if os.path.exists(hdf5_source):
        shutil.move(hdf5_source, hdf5_dest)
        print(f"Moved {hdf5_source} to {hdf5_dest}")
    else:
        print(f"Warning: Did not find expected output {hdf5_source}")

    # Move the YAML file
    if os.path.exists(yaml_source):
        shutil.move(yaml_source, yaml_dest)
        print(f"Moved {yaml_source} to {yaml_dest}")
    else:
        print(f"Warning: Did not find expected output {yaml_source}")    

print("Batch localization complete.")

In [5]:
import h5py
import numpy as np
import json

def load_info_as_json(info_group):
    # Picasso saves 'info' as a single-line JSON string in an HDF5 dataset
    info_str = info_group[()]
    if isinstance(info_str, bytes):  # Python 3 h5py reads as bytes
        info_str = info_str.decode()
    return json.loads(info_str)

def compare_hdf5(a, b):
    with h5py.File(a, 'r') as f1, h5py.File(b, 'r') as f2:
        # --- locs
        locs1 = f1['locs'][:]
        locs2 = f2['locs'][:]
        if locs1.shape != locs2.shape:
            print(f"Shape mismatch: {locs1.shape} vs {locs2.shape}")
            return False
        # For floats, use allclose for numerical safety
        identical = np.allclose(locs1, locs2, atol=1e-8)
        if not identical:
            print("Localization datasets differ!")
            return False
        else:
            print("Localization datasets match.")
        # --- info
        info1 = load_info_as_json(f1['info'])
        info2 = load_info_as_json(f2['info'])
        if info1 != info2:
            print("Info (metadata) differs:")
            print("info1:", info1)
            print("info2:", info2)
            return False
        else:
            print("Info (metadata) matches.")
    print("Files are identical in locs and info.")
    return True

# Usage:
f1 = '/Volumes/T9/DNA-PAINT/raw/localizations_manual/RawFramesforTraining_P5-IS_20pM_locs.hdf5'
f2 = '/Volumes/T9/DNA-PAINT/raw/localizations/RawFramesforTraining_P5-IS_20pM_locs.hdf5'
compare_hdf5(f1, f2)

Shape mismatch: (246983,) vs (246974,)


False

In [3]:
import yaml

def compare_yaml(a, b):
    with open(a) as fa, open(b) as fb:
        ya = list(yaml.safe_load_all(fa))
        yb = list(yaml.safe_load_all(fb))
    if ya == yb:
        print("YAML files are identical")
    else:
        print("YAML files differ!")
        # Optionally show diff:
        for i, (doca, docb) in enumerate(zip(ya, yb)):
            if doca != docb:
                print(f"Difference in document #{i+1}:")
                print("File A:", doca)
                print("File B:", docb)

yaml1 = '/Volumes/T9/DNA-PAINT/raw/localizations/RawFramesforTraining_P5-IS_20pM_locs.yaml'
yaml2 = '/Volumes/T9/DNA-PAINT/sparz_k11_rt5_lev0/uncompressed/localizations/NP5_RawFramesforTraining_P5-IS_20pM_compression_level_0_locs.yaml'
compare_yaml(yaml1, yaml2)

YAML files differ!
Difference in document #1:
File A: {'Byte Order': '<', 'Camera': 'Andor', 'Data Type': 'uint16', 'File': '/Volumes/T9/DNA-PAINT/raw/RawFramesforTraining_P5-IS_20pM.tif', 'Frames': 5000, 'Height': 512, 'Micro-Manager Acquisition Comments': [''], 'Micro-Manager Metadata': {'Andor-1. Camera Information : | Type | Model | Serial No. |': '| iXon | DU897_BV | 4102 |', 'Andor-ActualInterval-ms': '210.79', 'Andor-Advanced | Snap Image Additional Delay (ms)': '0', 'Andor-Advanced | Snap Image Timing Mode': 'Delay for Exposure', 'Andor-BaselineClamp': 'Enabled', 'Andor-Binning': '1', 'Andor-CCDTemperature': '-6', 'Andor-CCDTemperature Help': 'Wait for temperature to stabilize before acquisition.', 'Andor-CCDTemperature continuous update': 'Off', 'Andor-CCDTemperatureSetPoint': '-70', 'Andor-Camera': '| iXon | DU897_BV | 4102 |', 'Andor-CoolerMode': 'Fan off at shutdown', 'Andor-Description': 'Andor camera adapter', 'Andor-EMSwitch': 'On', 'Andor-Exposure': '150.00', 'Andor-Fan